# MindCare: Deterministic Crisis Detection & Safety Routing
This notebook demonstrates MindCare's deterministic safety net:
1. Multi-tier severity categorization (`high`, `medium`, `low`, `none`).
2. Dual-branch safety boundary (Harm-to-Self vs. Harm-to-Others / Violence).
3. Affective polarity suppression (suppressing contradictory positive emotion labels on violent inputs).


In [ ]:
import re
import json

# Crisis keyword taxonomy
CRISIS_KEYWORDS = {
    "high_self_harm": [
        "want to die", "end my life", "kill myself", "suicide", "suicidal",
        "no reason to live", "better off dead"
    ],
    "high_violence": [
        "kill someone", "kill somebody", "kill people", "want to kill",
        "murder someone", "murder somebody", "shoot someone", "harm others", "stab someone"
    ],
    "medium": [
        "hopeless", "can't go on", "cant go on", "give up", "worthless", "self harm", "self-harm"
    ],
    "low": [
        "overwhelmed", "panic attack", "can't cope", "cant cope", "breaking down"
    ]
}


## 1. Multi-Tier Severity Rule Engine


In [ ]:
def evaluate_crisis(text: str, ml_risk: str = "Normal") -> dict:
    text_lower = text.lower()
    
    # High crisis checks
    for kw in CRISIS_KEYWORDS["high_self_harm"]:
        if re.search(r'\b' + re.escape(kw) + r'\b', text_lower):
            return {"level": "high", "type": "self_harm", "keyword": kw, "bypass_gemini": True}
            
    for kw in CRISIS_KEYWORDS["high_violence"]:
        if re.search(r'\b' + re.escape(kw) + r'\b', text_lower):
            return {"level": "high", "type": "harm_to_others", "keyword": kw, "bypass_gemini": True}
            
    # ML Suicidal prediction check
    if ml_risk.lower() == "suicidal":
        return {"level": "high", "type": "self_harm", "keyword": "ml_flag", "bypass_gemini": True}
        
    # Medium crisis checks
    for kw in CRISIS_KEYWORDS["medium"]:
        if re.search(r'\b' + re.escape(kw) + r'\b', text_lower):
            return {"level": "medium", "type": "self_harm", "keyword": kw, "bypass_gemini": True}
            
    # Low risk checks
    for kw in CRISIS_KEYWORDS["low"]:
        if re.search(r'\b' + re.escape(kw) + r'\b', text_lower):
            return {"level": "low", "type": "none", "keyword": kw, "bypass_gemini": False}
            
    return {"level": "none", "type": "none", "keyword": None, "bypass_gemini": False}


## 2. Affective Polarity Suppression
Prevents contradictory emotion predictions like assigning 'Hopeful' or 'Joy' to violent or self-destructive intent.


In [ ]:
def apply_polarity_suppression(predicted_emotion: str, crisis_level: str) -> str:
    if crisis_level in {"high", "medium"} and predicted_emotion in {"Hopeful", "Joy", "Happy", "Calm"}:
        return "Sad"
    return predicted_emotion


## 3. Comprehensive Verification Test Cases


In [ ]:
test_cases = [
    ("I had a lovely, sunny afternoon with my friends.", "Normal", "Joy"),
    ("I feel completely overwhelmed with my exams and work.", "Anxiety", "Anxious"),
    ("I feel completely hopeless and like I cannot go on anymore.", "Depression", "Sad"),
    ("I want to end my life tonight.", "Suicidal", "Sad"),
    ("I am furious and I am going to hurt someone tomorrow.", "Normal", "Angry"),
    ("I am so happy I want to kill someone.", "Normal", "Joy") # Polarity test
]

print(f"{'Input Text':<55} | {'Risk':<10} | {'Crisis':<8} | {'Harm Type':<15} | {'Final Emotion':<10}")
print("-" * 110)

for text, ml_risk, raw_emo in test_cases:
    crisis_res = evaluate_crisis(text, ml_risk)
    final_emo = apply_polarity_suppression(raw_emo, crisis_res['level'])
    print(f"{text[:52]+'...':<55} | {ml_risk:<10} | {crisis_res['level']:<8} | {crisis_res['type']:<15} | {final_emo:<10}")
